In [1]:
import pandas as pd
import polars as pl
import numpy as np
from pathlib import Path

from polars import selectors as cs

schedule = Path('data/schedule')
statcast = Path('data/statcast')

In [25]:
sched_schema = pl.Schema(
    {
        'date': pl.Date,
        'total_items': pl.UInt8,
        'total_events': pl.UInt8,
        'total_games': pl.UInt8,
        'total_games_in_progress': pl.UInt8,
        'game_pk': pl.UInt32,
        'game_guid': pl.Categorical(),
        'link': pl.String,
        'game_type': pl.Categorical(),
        'season': pl.UInt16,
        'game_date': pl.Datetime(),
        'official_date': pl.Date,
        'is_tie': pl.Boolean,
        'game_number': pl.UInt8,
        'public_facing': pl.Boolean,
        'double_header': pl.Categorical(),
        'gameday_type': pl.Categorical(),
        'tiebreaker': pl.Boolean,
        'calendar_event_id': pl.String,
        'season_display': pl.UInt16,
        'day_night': pl.Categorical(),
        'description': pl.Categorical(),
        'scheduled_innings': pl.UInt8,
        'reverse_home_away_status': pl.Boolean,
        'inning_break_length': pl.UInt16,
        'games_in_series': pl.UInt8,
        'series_game_number': pl.UInt8,
        'series_description': pl.Categorical(),
        'record_source': pl.String,
        'if_necessary': pl.Boolean,
        'if_necessary_description': pl.Categorical(),
        'status_abstract_game_state': pl.Categorical(),
        'status_coded_game_state': pl.Categorical(),
        'status_detailed_state': pl.Categorical(),
        'status_status_code': pl.Categorical(),
        'status_start_time_tbd': pl.Boolean,
        'status_abstract_game_code': pl.Categorical(),
        'teams_away_score': pl.UInt8,
        'teams_away_is_winner': pl.Boolean,
        'teams_away_split_squad': pl.Boolean,
        'teams_away_series_number': pl.UInt8,
        'teams_away_team_id': pl.UInt16,
        'teams_away_team_name': pl.Categorical(),
        'teams_away_team_link': pl.String,
        'teams_away_league_record_wins': pl.UInt16,
        'teams_away_league_record_losses': pl.UInt16,
        'teams_away_league_record_ties': pl.UInt16,
        'teams_away_league_record_pct': pl.Float64,
        'teams_home_score': pl.UInt8,
        'teams_home_is_winner': pl.Boolean,
        'teams_home_split_squad': pl.Boolean,
        'teams_home_series_number': pl.UInt8,
        'teams_home_team_id': pl.UInt16,
        'teams_home_team_name': pl.Categorical(),
        'teams_home_team_link': pl.String,
        'teams_home_league_record_wins': pl.UInt16,
        'teams_home_league_record_losses': pl.UInt16,
        'teams_home_league_record_ties': pl.UInt16,
        'teams_home_league_record_pct': pl.Float64,
        'venue_id': pl.UInt32,
        'venue_name': pl.Categorical(),
        'venue_link': pl.String,
        'content_link': pl.String,
        'status_reason': pl.Categorical(),
        'reschedule_date': pl.Datetime(),
        'reschedule_game_date': pl.Date,
        'resume_date': pl.Datetime(),
        'resume_game_date': pl.Date,
        'resumed_from': pl.Datetime(),
        'resumed_from_date': pl.Date,
        'rescheduled_from': pl.Datetime(),
        'rescheduled_from_date': pl.Date,
        'events': pl.String
    }
)

In [26]:
sc_schema = pl.Schema(
    {
        'pitch_type': pl.Categorical(),
        'game_date': pl.Date,
        'release_speed': pl.Float64,
        'release_pos_x': pl.Float64,
        'release_pos_z': pl.Float64,
        'player_name': pl.String,
        'batter': pl.UInt32,
        'pitcher': pl.UInt32,
        'events': pl.Categorical(),
        'description': pl.Categorical(),
        'spin_dir': pl.UInt16,
        'spin_rate_deprecated': pl.UInt16,
        'break_angle_deprecated': pl.UInt16,
        'break_length_deprecated': pl.Float64,
        'zone': pl.Categorical(),
        'des': pl.String,
        'game_type': pl.Categorical(),
        'stand': pl.Categorical(),
        'p_throws': pl.Categorical(),
        'home_team': pl.Categorical(),
        'away_team': pl.Categorical(),
        'type': pl.Categorical(),
        'hit_location': pl.Categorical(),
        'bb_type': pl.Categorical(),
        'balls': pl.UInt8,
        'strikes': pl.UInt8,
        'game_year': pl.UInt16,
        'pfx_x': pl.Float64,
        'pfx_z': pl.Float64,
        'plate_x': pl.Float64,
        'plate_z': pl.Float64,
        'on_3b': pl.UInt32,
        'on_2b': pl.UInt32,
        'on_1b': pl.UInt32,
        'outs_when_up': pl.UInt8,
        'inning': pl.UInt8,
        'inning_topbot': pl.Categorical(),
        'hc_x': pl.Float64,
        'hc_y': pl.Float64,
        'tfs_deprecated': pl.String,
        'tfs_zulu_deprecated': pl.String,
        'umpire': pl.UInt32,
        'sv_id': pl.String,
        'vx0': pl.Float64,
        'vy0': pl.Float64,
        'vz0': pl.Float64,
        'ax': pl.Float64,
        'ay': pl.Float64,
        'az': pl.Float64,
        'sz_top': pl.Float64,
        'sz_bot': pl.Float64,
        'hit_distance_sc': pl.UInt16,
        'launch_speed': pl.Float64,
        'launch_angle': pl.Int16,
        'effective_speed': pl.Float64,
        'release_spin_rate': pl.UInt16,
        'release_extension': pl.Float64,
        'game_pk': pl.UInt32,
        'fielder_2': pl.UInt32,
        'fielder_3': pl.UInt32,
        'fielder_4': pl.UInt32,
        'fielder_5': pl.UInt32,
        'fielder_6': pl.UInt32,
        'fielder_7': pl.UInt32,
        'fielder_8': pl.UInt32,
        'fielder_9': pl.UInt32,
        'release_pos_y': pl.Float64,
        'estimated_ba_using_speedangle': pl.Float64,
        'estimated_woba_using_speedangle': pl.Float64,
        'woba_value': pl.Float64,
        'woba_denom': pl.UInt8,
        'babip_value': pl.UInt8,
        'iso_value': pl.UInt8,
        'launch_speed_angle': pl.UInt8,
        'at_bat_number': pl.UInt16,
        'pitch_number': pl.UInt8,
        'pitch_name': pl.Categorical(),
        'home_score': pl.UInt8,
        'away_score': pl.UInt8,
        'bat_score': pl.UInt8,
        'fld_score': pl.UInt8,
        'post_away_score': pl.UInt8,
        'post_home_score': pl.UInt8,
        'post_bat_score': pl.UInt8,
        'post_fld_score': pl.UInt8,
        'if_fielding_alignment': pl.Categorical(),
        'of_fielding_alignment': pl.Categorical(),
        'spin_axis': pl.UInt16,
        'delta_home_win_exp': pl.Float64,
        'delta_run_exp': pl.Float64,
        'bat_speed': pl.Float64,
        'swing_length': pl.Float64,
        'estimated_slg_using_speedangle': pl.Float64,
        'delta_pitcher_run_exp': pl.Float64,
        'hyper_speed': pl.Float64,
        'home_score_diff': pl.Int8,
        'bat_score_diff': pl.Int8,
        'home_win_exp': pl.Float64,
        'bat_win_exp': pl.Float64,
        'age_pit_legacy': pl.UInt8,
        'age_bat_legacy': pl.UInt8,
        'age_pit': pl.UInt8,
        'age_bat': pl.UInt8,
        'n_thruorder_pitcher': pl.UInt8,
        'n_priorpa_thisgame_player_at_bat': pl.UInt8,
        'pitcher_days_since_prev_game': pl.UInt8,
        'batter_days_since_prev_game': pl.UInt8,
        'pitcher_days_until_next_game': pl.UInt8,
        'batter_days_until_next_game': pl.UInt8,
        'api_break_z_with_gravity': pl.Float64,
        'api_break_x_arm': pl.Float64,
        'api_break_x_batter_in': pl.Float64,
        'arm_angle': pl.Float64,
        'attack_angle': pl.Float64,
        'attack_direction': pl.Float64,
        'swing_path_tilt': pl.Float64,
        'intercept_ball_minus_batter_pos_x_inches': pl.Float64,
        'intercept_ball_minus_batter_pos_y_inches': pl.Float64,
    }
)

In [27]:
sc_schema_clean = pl.Schema(
    {
        'pitch_type': pl.Categorical(),
        'game_date': pl.Date,
        'release_speed': pl.Float64,
        'release_pos_x': pl.Float64,
        'release_pos_z': pl.Float64,
        'player_name': pl.String,
        'batter': pl.UInt32,
        'pitcher': pl.UInt32,
        'events': pl.Categorical(),
        'description': pl.Categorical(),
        'zone': pl.Categorical(),
        'des': pl.String,
        'game_type': pl.Categorical(),
        'stand': pl.Categorical(),
        'p_throws': pl.Categorical(),
        'home_team': pl.Categorical(),
        'away_team': pl.Categorical(),
        'type': pl.Categorical(),
        'hit_location': pl.Categorical(),
        'bb_type': pl.Categorical(),
        'balls': pl.UInt8,
        'strikes': pl.UInt8,
        'game_year': pl.UInt16,
        'pfx_x': pl.Float64,
        'pfx_z': pl.Float64,
        'plate_x': pl.Float64,
        'plate_z': pl.Float64,
        'on_3b': pl.UInt32,
        'on_2b': pl.UInt32,
        'on_1b': pl.UInt32,
        'outs_when_up': pl.UInt8,
        'inning': pl.UInt8,
        'inning_topbot': pl.Categorical(),
        'hc_x': pl.Float64,
        'hc_y': pl.Float64,
        'vx0': pl.Float64,
        'vy0': pl.Float64,
        'vz0': pl.Float64,
        'ax': pl.Float64,
        'ay': pl.Float64,
        'az': pl.Float64,
        'sz_top': pl.Float64,
        'sz_bot': pl.Float64,
        'hit_distance_sc': pl.UInt16,
        'launch_speed': pl.Float64,
        'launch_angle': pl.Int16,
        'effective_speed': pl.Float64,
        'release_spin_rate': pl.UInt16,
        'release_extension': pl.Float64,
        'game_pk': pl.UInt32,
        'fielder_2': pl.UInt32,
        'fielder_3': pl.UInt32,
        'fielder_4': pl.UInt32,
        'fielder_5': pl.UInt32,
        'fielder_6': pl.UInt32,
        'fielder_7': pl.UInt32,
        'fielder_8': pl.UInt32,
        'fielder_9': pl.UInt32,
        'release_pos_y': pl.Float64,
        'estimated_ba_using_speedangle': pl.Float64,
        'estimated_woba_using_speedangle': pl.Float64,
        'woba_value': pl.Float64,
        'woba_denom': pl.UInt8,
        'babip_value': pl.UInt8,
        'iso_value': pl.UInt8,
        'launch_speed_angle': pl.UInt8,
        'at_bat_number': pl.UInt16,
        'pitch_number': pl.UInt8,
        'pitch_name': pl.Categorical(),
        'home_score': pl.UInt8,
        'away_score': pl.UInt8,
        'bat_score': pl.UInt8,
        'fld_score': pl.UInt8,
        'post_away_score': pl.UInt8,
        'post_home_score': pl.UInt8,
        'post_bat_score': pl.UInt8,
        'post_fld_score': pl.UInt8,
        'if_fielding_alignment': pl.Categorical(),
        'of_fielding_alignment': pl.Categorical(),
        'spin_axis': pl.UInt16,
        'delta_home_win_exp': pl.Float64,
        'delta_run_exp': pl.Float64,
        'bat_speed': pl.Float64,
        'swing_length': pl.Float64,
        'estimated_slg_using_speedangle': pl.Float64,
        'delta_pitcher_run_exp': pl.Float64,
        'hyper_speed': pl.Float64,
        'home_score_diff': pl.Int8,
        'bat_score_diff': pl.Int8,
        'home_win_exp': pl.Float64,
        'bat_win_exp': pl.Float64,
        'age_pit_legacy': pl.UInt8,
        'age_bat_legacy': pl.UInt8,
        'age_pit': pl.UInt8,
        'age_bat': pl.UInt8,
        'n_thruorder_pitcher': pl.UInt8,
        'n_priorpa_thisgame_player_at_bat': pl.UInt8,
        'pitcher_days_since_prev_game': pl.UInt8,
        'batter_days_since_prev_game': pl.UInt8,
        'pitcher_days_until_next_game': pl.UInt8,
        'batter_days_until_next_game': pl.UInt8,
        'api_break_z_with_gravity': pl.Float64,
        'api_break_x_arm': pl.Float64,
        'api_break_x_batter_in': pl.Float64,
        'arm_angle': pl.Float64,
        'attack_angle': pl.Float64,
        'attack_direction': pl.Float64,
        'swing_path_tilt': pl.Float64,
        'intercept_ball_minus_batter_pos_x_inches': pl.Float64,
        'intercept_ball_minus_batter_pos_y_inches': pl.Float64,
    }
)


In [95]:
pbp_schema = pl.Schema(
    {
        'game_pk': pl.UInt32,
        'game_date': pl.Date,
        'index': pl.UInt8,
        'startTime': pl.Datetime(),
        'endTime': pl.Datetime(),
        'isPitch': pl.Boolean,
        'type': pl.Categorical(),
        'playId': pl.String,
        'pitchNumber': pl.UInt8,
        'details.description': pl.String,
        'details.event': pl.Categorical(),
        'details.awayScore': pl.UInt8,
        'details.homeScore': pl.UInt8,
        'details.isScoringPlay': pl.Boolean,
        'details.hasReview': pl.Boolean,
        'details.code': pl.Categorical(),
        'details.ballColor': pl.String,
        'details.isInPlay': pl.Boolean,
        'details.isStrike': pl.Boolean,
        'details.isBall': pl.Boolean,
        'details.call.code': pl.Categorical(),
        'details.call.description': pl.Categorical(),
        'count.balls.start': pl.UInt8,
        'count.strikes.start': pl.UInt8,
        'count.outs.start': pl.UInt8,
        'player.id': pl.UInt32,
        'player.link': pl.String,
        'pitchData.strikeZoneTop': pl.Float64,
        'pitchData.strikeZoneBottom': pl.Float64,
        'details.fromCatcher': pl.Boolean,
        'pitchData.coordinates.x': pl.Float64,
        'pitchData.coordinates.y': pl.Float64,
        'hitData.trajectory': pl.Categorical(),
        'hitData.hardness': pl.Categorical(),
        'hitData.location': pl.Categorical(),
        'hitData.coordinates.coordX': pl.Float64,
        'hitData.coordinates.coordY': pl.Float64,
        'actionPlayId': pl.String,
        'details.eventType': pl.Categorical(),
        'details.runnersGoing': pl.Boolean,
        'position.code': pl.Categorical(),
        'position.name': pl.Categorical(),
        'position.type': pl.Categorical(),
        'position.abbreviation': pl.Categorical(),
        'battingOrder': pl.Categorical(),
        'at_BatIndex': pl.UInt16,
        'result.type': pl.Categorical(),
        'result.event': pl.Categorical(),
        'results.eventType': pl.Categorical(),
        'result.description': pl.String,
        'result.rbi': pl.UInt8,
        'result.awayScore': pl.UInt8,
        'result.homeScore': pl.UInt8,
        'about.atBatIndex': pl.UInt16,
        'about.halfInning': pl.Categorical(),
        'about.inning': pl.UInt8,
        'about.startTime': pl.Datetime(),
        'about.endTime': pl.Datetime(),
        'about.isCompleted': pl.Boolean,
        'about.isScoringPlay': pl.Boolean,
        'about.hasReview': pl.Boolean,
        'about.hasOut': pl.Boolean,
        'about.captivatingIndex': pl.UInt8,
        'count.balls.end': pl.UInt8,
        'count.strikes.end': pl.UInt8,
        'count.outs.end': pl.UInt8,
        'mathchup.batter.id': pl.UInt32,
        'mathcup.batter.fullName': pl.String,
        'mathcup.batter.link': pl.String,
        'matchup.batSide.code': pl.Categorical(),
        'matchup.batSide.description': pl.Categorical(),
        'matchup.pitcher.id': pl.UInt32,
        'matchup.pitcher.fullName': pl.String,
        'matchup.pitcher.link': pl.String,
        'matchup.pitchHand.code': pl.Categorical(),
        'matchup.pitchHand.description': pl.Categorical(),
        'mathcup.splits.batter': pl.Categorical(),
        'matchup.splits.pitcher': pl.Categorical(),
        'matchup.splits.menOnBase': pl.Categorical(),
        'batted.ball.result': pl.Categorical(),
        'home_team': pl.Categorical(),
        'home_level_id': pl.UInt8,
        'home_level_name': pl.Categorical(),
        'home_parentOrg_id': pl.UInt16,
        'home_parentOrg_name': pl.Categorical(),
        'home_league_id': pl.UInt16,
        'home_league_name': pl.Categorical(),
        'away_team': pl.Categorical(),
        'away_level_id': pl.UInt8,
        'away_level_name': pl.Categorical(),
        'away_parentOrg_id': pl.UInt16,
        'away_parentOrg_name': pl.Categorical(),
        'away_league_id': pl.UInt16,
        'away_league_name': pl.Categorical(),
        'batting_team': pl.Categorical(),
        'fielding_team': pl.Categorical(),
        'last.pitch.of.ab': pl.Boolean,
        'pfxId': pl.UInt8,
        'details.trailColor': pl.String,
        'details.type.description': pl.Categorical(),
        'pitchData.startSpeed': pl.Float64,
        'pitchData.endSpeed': pl.Float64,
        'pitchData.zone': pl.UInt8,
        'pitchData.typeConf': pl.Float64,
        'pitchData.plateTime': pl.Float64,
        'pitchData.extension': pl.Float64,
        'pitchData.coordinates.aY': pl.Float64,
        'pitchData.coordinates.aZ': pl.Float64,
        'pitchData.coordinates.pfxX': pl.Float64,
        'pitchData.coordinates.pfxZ': pl.Float64,
        'pitchData.coordinates.pX': pl.Float64,
        'pitchData.coordinates.pZ': pl.Float64,
        'pitchData.coordinates.vX0': pl.Float64,
        'pitchData.coordinates.vY0': pl.Float64,
        'pitchData.coordinates.vZ0': pl.Float64,
        'pitchData.coordinates.x0': pl.Float64,
        'pitchData.coordinates.y0': pl.Float64,
        'pitchData.coordinates.z0': pl.Float64,
        'pitchData.coordinates.aX': pl.Float64,
        'pitchData.breakAngle': pl.Float64,
        'pitchData.breakLength': pl.Float64,
        'pitchData.breaks.breakY': pl.Float64,
        'pitchData.breaks.spinRate': pl.UInt16,
        'pitchdata.breaks.spinDirection': pl.UInt16,
        'hitData.launchSpeed': pl.Float64,
        'hitdata.launchAngle': pl.Int16,
        'hitData.totalDistance': pl.Int16,
        'injuryType': pl.Categorical(),
        'umpire.id': pl.UInt32,
        'umpire.link': pl.String,
        'details.isOut': pl.Boolean,
        'isBaseRunningPlay': pl.Boolean,
        'isSubstitution': pl.Boolean,
        'base': pl.UInt8,
        'details.disengagementNum': pl.UInt8,
        'replacedPlayer.id': pl.UInt32,
        'replacedPlayer.link': pl.String,
        'result.isOut': pl.Boolean,
        'about.isTopInning': pl.Boolean,
        'matchup.postOnFirst.id': pl.UInt32,
        'matchup.postOnFirst.fullName': pl.String,
        'matchup.postOnFirst.link': pl.String,
        'mathcup.postOnThird.id': pl.UInt32,
        'mathcup.postOnThird.fullName': pl.String,
        'mathcup.postOnThird.link': pl.String,
        'mathcup.postOnSecond.id': pl.UInt32,
        'mathcup.postOnSecond.fullName': pl.String,
        'mathcup.postOnSecond.link': pl.String,
        'pitchData.breaks.breakVertical': pl.Float64,
        'pitchData.breaks.breakVerticalInduced': pl.Float64,
        'pitchData.breaks.breakHorizontal': pl.Float64,
        'reviewDetails.isOverturnedd': pl.Boolean,
        'reviewDetails.inProgress': pl.Boolean,
        'reviewDetails.reviewType': pl.Categorical(),
        'reviewDetails.challengeTeamId': pl.UInt16,
        'reviewDetails.isOverturened.x': pl.Boolean,
        'reviewDetails.inProgress.x': pl.Boolean,
        'reviewDetails.reviewType.x': pl.Categorical(),
        'reviewDetails.challengeTeamId.x': pl.UInt16,
        'reviewDetails.isOverturned.y': pl.Boolean,
        'reviewDetails.inProgress.y': pl.Boolean,
        'reviewDetails.reviewType.y': pl.Categorical(),
        'reviewDetails.challengeTeamId.y': pl.UInt16,
        'reviewDetails.additionalReviews': pl.Boolean

    }
)

In [96]:
print(len(pbp_schema.names()))
print(len(df_pbp.columns))
[col for col in df_pbp.columns if col not in pbp_schema.names()]

164
165


C:\Users\theru\AppData\Local\Temp\ipykernel_12572\3771426806.py:2: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  print(len(df_pbp.columns))
C:\Users\theru\AppData\Local\Temp\ipykernel_12572\3771426806.py:3: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  [col for col in df_pbp.columns if col not in pbp_schema.names()]


['atBatIndex',
 'result.eventType',
 'about.isComplete',
 'matchup.batter.id',
 'matchup.batter.fullName',
 'matchup.batter.link',
 'matchup.splits.batter',
 'details.type.code',
 'pitchData.typeConfidence',
 'pitchData.breaks.breakAngle',
 'pitchData.breaks.breakLength',
 'pitchData.breaks.spinDirection',
 'hitData.launchAngle',
 'matchup.postOnThird.id',
 'matchup.postOnThird.fullName',
 'matchup.postOnThird.link',
 'matchup.postOnSecond.id',
 'matchup.postOnSecond.fullName',
 'matchup.postOnSecond.link',
 'reviewDetails.isOverturned',
 'reviewDetails.isOverturned.x']

In [88]:
df_pbp = pl.scan_csv(
    'data/pbp/pbp_2017.csv',
    null_values='NA'
)
df_pbp.describe()

statistic,game_pk,game_date,index,startTime,endTime,isPitch,type,playId,pitchNumber,details.description,details.event,details.awayScore,details.homeScore,details.isScoringPlay,details.hasReview,details.code,details.ballColor,details.isInPlay,details.isStrike,details.isBall,details.call.code,details.call.description,count.balls.start,count.strikes.start,count.outs.start,player.id,player.link,pitchData.strikeZoneTop,pitchData.strikeZoneBottom,details.fromCatcher,pitchData.coordinates.x,pitchData.coordinates.y,hitData.trajectory,hitData.hardness,hitData.location,hitData.coordinates.coordX,…,injuryType,umpire.id,umpire.link,details.isOut,isBaseRunningPlay,isSubstitution,base,details.disengagementNum,replacedPlayer.id,replacedPlayer.link,result.isOut,about.isTopInning,matchup.postOnFirst.id,matchup.postOnFirst.fullName,matchup.postOnFirst.link,matchup.postOnThird.id,matchup.postOnThird.fullName,matchup.postOnThird.link,matchup.postOnSecond.id,matchup.postOnSecond.fullName,matchup.postOnSecond.link,pitchData.breaks.breakVertical,pitchData.breaks.breakVerticalInduced,pitchData.breaks.breakHorizontal,reviewDetails.isOverturned,reviewDetails.inProgress,reviewDetails.reviewType,reviewDetails.challengeTeamId,reviewDetails.isOverturned.x,reviewDetails.inProgress.x,reviewDetails.reviewType.x,reviewDetails.challengeTeamId.x,reviewDetails.isOverturned.y,reviewDetails.inProgress.y,reviewDetails.reviewType.y,reviewDetails.challengeTeamId.y,reviewDetails.additionalReviews
str,f64,str,f64,str,str,f64,str,str,f64,str,str,f64,f64,f64,f64,str,str,f64,f64,f64,str,str,f64,f64,f64,f64,str,f64,f64,str,f64,f64,str,str,f64,str,…,str,str,str,f64,str,f64,str,str,f64,str,f64,f64,f64,str,str,f64,str,str,f64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""count""",925841.0,"""925841""",925841.0,"""925792""","""925792""",925841.0,"""925841""","""847872""",828733.0,"""925493""","""76710""",76710.0,76710.0,76710.0,925841.0,"""849131""","""825242""",828733.0,828733.0,828733.0,"""828733""","""828733""",925841.0,925841.0,925841.0,76710.0,"""76710""",825242.0,825242.0,"""20051""",825177.0,825199.0,"""158808""","""158818""",158228.0,"""153685""",…,"""362""","""235""","""235""",925841.0,"""7285""",46167.0,"""1833""","""44430""",31917.0,"""31917""",925841.0,925841.0,264603.0,"""264603""","""264603""",26022.0,"""26022""","""26022""",81595.0,"""81595""","""81595""","""743564""","""743564""","""743564""","""1145""","""1145""","""1139""","""1048""","""105""","""105""","""104""","""76""","""116""","""116""","""115""","""87""","""0"""
"""null_count""",0.0,"""0""",0.0,"""49""","""49""",0.0,"""0""","""77969""",97108.0,"""348""","""849131""",849131.0,849131.0,849131.0,0.0,"""76710""","""100599""",97108.0,97108.0,97108.0,"""97108""","""97108""",0.0,0.0,0.0,849131.0,"""849131""",100599.0,100599.0,"""905790""",100664.0,100642.0,"""767033""","""767023""",767613.0,"""772156""",…,"""925479""","""925606""","""925606""",0.0,"""918556""",879674.0,"""924008""","""881411""",893924.0,"""893924""",0.0,0.0,661238.0,"""661238""","""661238""",899819.0,"""899819""","""899819""",844246.0,"""844246""","""844246""","""182277""","""182277""","""182277""","""924696""","""924696""","""924702""","""924793""","""925736""","""925736""","""925737""","""925765""","""925725""","""925725""","""925726""","""925754""","""925841"""
"""mean""",494069.699451,null,2.106855,null,null,0.891343,null,null,2.81495,null,null,3.040282,3.034715,0.006857,0.000305,null,null,0.19164,0.449881,0.35848,null,null,1.11215,1.131024,0.947493,544172.025212,null,3.43795,1.569341,null,114.576771,174.87947,null,null,7.667549,null,…,null,null,null,0.167586,null,1.0,null,null,529841.612934,null,0.662101,0.513387,531531.048405,null,null,535443.828799,null,null,532826.919885,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""std""",7163.368878,null,1.958664,null,null,null,null,null,1.721711,null,null,2.939352,2.978959,null,null,null,null,null,null,null,null,null,1.063734

In [75]:
df_pbp.filter(
    ~pl.col('battingOrder').is_null()
).collect().sample(1000)

game_pk,game_date,index,startTime,endTime,isPitch,type,playId,pitchNumber,details.description,details.event,details.awayScore,details.homeScore,details.isScoringPlay,details.hasReview,details.code,details.ballColor,details.isInPlay,details.isStrike,details.isBall,details.call.code,details.call.description,count.balls.start,count.strikes.start,count.outs.start,player.id,player.link,pitchData.strikeZoneTop,pitchData.strikeZoneBottom,details.fromCatcher,pitchData.coordinates.x,pitchData.coordinates.y,hitData.trajectory,hitData.hardness,hitData.location,hitData.coordinates.coordX,hitData.coordinates.coordY,…,injuryType,umpire.id,umpire.link,details.isOut,isBaseRunningPlay,isSubstitution,base,details.disengagementNum,replacedPlayer.id,replacedPlayer.link,result.isOut,about.isTopInning,matchup.postOnFirst.id,matchup.postOnFirst.fullName,matchup.postOnFirst.link,matchup.postOnThird.id,matchup.postOnThird.fullName,matchup.postOnThird.link,matchup.postOnSecond.id,matchup.postOnSecond.fullName,matchup.postOnSecond.link,pitchData.breaks.breakVertical,pitchData.breaks.breakVerticalInduced,pitchData.breaks.breakHorizontal,reviewDetails.isOverturned,reviewDetails.inProgress,reviewDetails.reviewType,reviewDetails.challengeTeamId,reviewDetails.isOverturned.x,reviewDetails.inProgress.x,reviewDetails.reviewType.x,reviewDetails.challengeTeamId.x,reviewDetails.isOverturned.y,reviewDetails.inProgress.y,reviewDetails.reviewType.y,reviewDetails.challengeTeamId.y,reviewDetails.additionalReviews
i64,str,i64,str,str,bool,str,str,i64,str,str,i64,i64,bool,bool,str,str,bool,bool,bool,str,str,i64,i64,i64,i64,str,f64,f64,str,f64,f64,str,str,i64,str,str,…,str,str,str,bool,str,bool,str,str,i64,str,bool,bool,i64,str,str,i64,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
509505,"""2017-02-22""",1,"""2017-02-22T21:16:04.192Z""","""2017-02-22T21:16:04.220Z""",false,"""action""",null,null,"""Pitching Change: Jared Miller …","""Pitching Substitution""",0,6,false,false,null,null,null,null,null,null,null,0,0,0,656744,"""/api/v1/people/656744""",null,null,null,null,null,null,null,null,null,null,…,null,null,null,false,null,true,null,null,542455,"""/api/v1/people/542455""",true,true,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
509505,"""2017-02-22""",3,"""2017-02-22T21:16:04.245Z""","""2017-02-22T21:16:04.247Z""",false,"""action""",null,null,"""Defensive Substitution: Jason …","""Defensive Sub""",0,6,false,false,null,null,null,null,null,null,null,0,0,0,445095,"""/api/v1/people/445095""",null,null,null,null,null,null,null,null,null,null,…,null,null,null,false,null,true,null,null,606466,"""/api/v1/people/606466""",true,true,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
509663,"""2017-02-23""",1,"""2017-02-23T19:43:54.752Z""","""2017-02-23T19:44:02.094Z""",false,"""action""",null,null,"""Offensive Substitution: Pinch-…","""Offensive Substitution""",0,6,false,false,null,null,null,null,null,null,null,0,0,2,623510,"""/api/v1/people/623510""",null,null,null,null,null,null,null,null,null,null,…,null,null,null,false,null,true,"""1""",null,519068,"""/api/v1/people/519068""",false,false,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
509663,"""2017-02-23""",2,"""2017-02-23T18:54:52.023Z""","""2017-02-23T18:56:05.488Z""",false,"""action""",null,null,"""Defensive Substitution: Miguel…","""Defensive Sub""",0,4,false,false,null,null,null,null,null,null,null,0,0,0,544838,"""/api/v1/people/544838""",null,null,null,null,null,null,null,null,null,null,…,null,null,null,false,null,true,null,null,543510,"""/api/v1/people/543510""",true,true,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
509664,"""2017-02-24""",3,"""2017-02-24T22:17:44.745Z""","""20

In [28]:
def clean_schedules(schema):
    for f in schedule.glob('schedule_????.csv'):
        df = pd.read_csv(f)
        map_dict = {'N': False, 'Y': True}

        df['tiebreaker'] = df['tiebreaker'].map(map_dict)

        df['if_necessary'] = df['if_necessary'].map(map_dict)
        for col, dtype in zip(sched_schema.names(), sched_schema.dtypes()):
            if col not in df.columns:
                df[col] = np.nan

            if dtype == pl.UInt8:
                df[col] = df[col].astype('Int64')
            elif dtype == pl.UInt16:
                df[col] = df[col].astype('Int64')
            elif dtype == pl.UInt32:
                df[col] = df[col].astype('Int64')
            elif dtype == pl.UInt64:
                df[col] = df[col].astype('Int64')
        df[schema.names()].to_csv(schedule/('clean_'+f.name), index=False)

        df = pl.scan_csv(schedule/('clean_'+f.name))
        df = df.filter(
            ~pl.col('is_tie').is_null()
        ).with_columns(
            pl.col('description').fill_null('no description'),
            pl.col('inning_break_length').fill_null(strategy='backward'),
            pl.col('games_in_series').fill_null(strategy='zero'),
            pl.col('series_game_number').fill_null(strategy='zero'),
            pl.col('teams_away_is_winner').fill_null(False),
            pl.col('teams_home_is_winner').fill_null(False),
            pl.col('teams_away_series_number').fill_null(strategy='zero'),
            pl.col('teams_home_series_number').fill_null(strategy='zero'),
        )
        df.collect().write_csv(schedule/('clean_'+f.name))
clean_schedules(sched_schema)

In [29]:
def clean_statcast(schema):
    for f in statcast.glob('statcast_????.csv'):
        df = pl.scan_csv(f, schema = schema)
        df = df.with_columns(
            pl.col('events').fill_null(pl.col('description')),
            pl.col('des').fill_null(pl.col('description')),
            pl.when(
                pl.col('hit_location').is_null() &
                (pl.col('events') == 'home_run')
            ).then(
                pl.col('hit_location').fill_null("HR"),
            ).when(
                pl.col('hit_location').is_null() &
                ((pl.col('events') == 'double') & pl.col('des').str.contains('ground-rule double'))
            ).then(
                pl.col('hit_location').fill_null("GRD"),
            ).when(
                pl.col('hit_location').is_null() &
                pl.col('des').str.contains('fan interference') &
                ~pl.col('des').str.contains('ground-rule double')
            ).then(
                pl.col('hit_location').fill_null("FAN"),
            ).when(
                pl.col('hit_location').is_null() &
                (pl.col('type') == 'X')
            ).then(
                pl.col('hit_location').fill_null("UNKNOWN"),
            ).otherwise(
                pl.col('hit_location').fill_null("NO_HIT")
            ),
            pl.col('bb_type').fill_null("not_in_play"),
            cs.matches('on_[1-3]b').fill_null(strategy='zero'),
        ).select(
            pl.exclude(
                'spin_dir',
                'spin_rate_deprecated',
                'break_angle_deprecated',
                'break_length_deprecated',
                'tfs_zulu_deprecated',
                'tfs_deprecated',
                'umpire',
                'sv_id'
            )
        )

        df.collect().write_csv(statcast/('clean_'+f.name))
clean_statcast(sc_schema)

In [35]:
df_sch = pl.scan_csv(schedule/'clean_schedule_????.csv', schema = sched_schema)
df_sc = pl.scan_csv(statcast/'clean_statcast_????.csv', schema = sc_schema_clean)

In [31]:
df_sc.describe()

statistic,pitch_type,game_date,release_speed,release_pos_x,release_pos_z,player_name,batter,pitcher,events,description,zone,des,game_type,stand,p_throws,home_team,away_team,type,hit_location,bb_type,balls,strikes,game_year,pfx_x,pfx_z,plate_x,plate_z,on_3b,on_2b,on_1b,outs_when_up,inning,inning_topbot,hc_x,hc_y,vx0,…,post_away_score,post_home_score,post_bat_score,post_fld_score,if_fielding_alignment,of_fielding_alignment,spin_axis,delta_home_win_exp,delta_run_exp,bat_speed,swing_length,estimated_slg_using_speedangle,delta_pitcher_run_exp,hyper_speed,home_score_diff,bat_score_diff,home_win_exp,bat_win_exp,age_pit_legacy,age_bat_legacy,age_pit,age_bat,n_thruorder_pitcher,n_priorpa_thisgame_player_at_bat,pitcher_days_since_prev_game,batter_days_since_prev_game,pitcher_days_until_next_game,batter_days_until_next_game,api_break_z_with_gravity,api_break_x_arm,api_break_x_batter_in,arm_angle,attack_angle,attack_direction,swing_path_tilt,intercept_ball_minus_batter_pos_x_inches,intercept_ball_minus_batter_pos_y_inches
str,str,str,f64,f64,f64,str,f64,f64,str,str,str,str,str,str,str,str,str,str,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,f64,f64,f64,…,f64,f64,f64,f64,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""count""","""7662949""","""7799380""",7.665475e6,7.664602e6,7.664602e6,"""7799380""",7.79938e6,7.79938e6,"""7799380""","""7799380""","""7664322""","""7799380""","""7799380""","""7799380""","""7799380""","""7799380""","""7799380""","""7799380""","""7799380""","""7799380""",7.79938e6,7.79938e6,7.79938e6,7.664607e6,7.664751e6,7.664322e6,7.664322e6,7.79938e6,7.79938e6,7.79938e6,7.79938e6,7.79938e6,"""7799380""",1.357832e6,1.357832e6,7.664812e6,…,7.79938e6,7.79938e6,7.79938e6,7.79938e6,"""7581862""","""7581862""",6.923909e6,7.799348e6,7.769841e6,811081.0,811081.0,1.291065e6,7.769841e6,2.290562e6,7.79938e6,7.79938e6,7.79935e6,7.79935e6,7.79938e6,7.79938e6,7.79938e6,7.79938e6,7.79938e6,7.79938e6,7.207525e6,7.46241e6,7.236308e6,7.469149e6,7.664604e6,7.664607e6,7.664607e6,3.786564e6,811081.0,811081.0,811075.0,810175.0,810175.0
"""null_count""","""136431""","""0""",133905.0,134778.0,134778.0,"""0""",0.0,0.0,"""0""","""0""","""135058""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""",0.0,0.0,0.0,134773.0,134629.0,135058.0,135058.0,0.0,0.0,0.0,0.0,0.0,"""0""",6.441548e6,6.441548e6,134568.0,…,0.0,0.0,0.0,0.0,"""217518""","""217518""",875471.0,32.0,29539.0,6.988299e6,6.988299e6,6.508315e6,29539.0,5.508818e6,0.0,0.0,30.0,30.0,0.0,0.0,0.0,0.0,0.0,0.0,591855.0,336970.0,563072.0,330231.0,134776.0,134773.0,134773.0,4.012816e6,6.988299e6,6.988299e6,6.988305e6,6.989205e6,6.989205e6
"""mean""",null,"""2020-07-31 08:24:19.807112""",88.868149,-0.793726,5.863429,null,581309.795305,582378.317676,null,null,null,null,null,null,null,null,null,null,null,null,0.87748,0.887378,2020.080001,-0.125932,0.652869,0.028987,2.27276,55557.999415,109682.229661,176925.861167,0.982177,4.977218,null,126.678492,122.958997,2.343546,…,2.36213,2.230364,2.292245,2.300249,null,null,178.014429,0.000257,0.000148,69.588397,7.214767,0.544226,-0.000148,91.886702,-0.132085,-0.038833,0.508062,0.513071,28.504961,28.13189,29.026521,28.663105,1.499132,1.515888,5.815862,1.757308,5.927575,1.747634,2.300268,0.376953,-0.113606,38.874974,9.160892,-0.676799,32.332275,37.08873,29.905032
"""std""",null,null,6.058157,1.895897,0.521578,null,85648.870192,82087.405421,null,null,null,null,null,null,null,null,null,null,null,null,0.967923,0.827612,3.242158,0.885809,0.743075,0.860721,0.962367,173183.027611,230889.490157,271649.913661,0.817723,2.632737,null,40.250585,42.304868,5.952344,…,2.64996,2.586684,2.580776,2.657341,null,null,70.552226,0.028289,0.227766,8.973566,1.000515,0.698952,0.227766,6.133604,3.191323,3.193819,0.293245,0.293065,3.736695,3.73555,3.744963,3.757125,0.727544,1.272633,7.572445,4.029154,7.909412,4.024197,1.11242,0.811432,0.887474,12.931691,11.800878,20.48010

In [48]:
df_sc.filter(
    pl.any_horizontal('launch_speed','launch_angle', 'bat_speed', 'hit_distance_sc').is_null() &
    pl.col('type').eq('X')
).describe()

statistic,pitch_type,game_date,release_speed,release_pos_x,release_pos_z,player_name,batter,pitcher,events,description,zone,des,game_type,stand,p_throws,home_team,away_team,type,hit_location,bb_type,balls,strikes,game_year,pfx_x,pfx_z,plate_x,plate_z,on_3b,on_2b,on_1b,outs_when_up,inning,inning_topbot,hc_x,hc_y,vx0,…,post_away_score,post_home_score,post_bat_score,post_fld_score,if_fielding_alignment,of_fielding_alignment,spin_axis,delta_home_win_exp,delta_run_exp,bat_speed,swing_length,estimated_slg_using_speedangle,delta_pitcher_run_exp,hyper_speed,home_score_diff,bat_score_diff,home_win_exp,bat_win_exp,age_pit_legacy,age_bat_legacy,age_pit,age_bat,n_thruorder_pitcher,n_priorpa_thisgame_player_at_bat,pitcher_days_since_prev_game,batter_days_since_prev_game,pitcher_days_until_next_game,batter_days_until_next_game,api_break_z_with_gravity,api_break_x_arm,api_break_x_batter_in,arm_angle,attack_angle,attack_direction,swing_path_tilt,intercept_ball_minus_batter_pos_x_inches,intercept_ball_minus_batter_pos_y_inches
str,str,str,f64,f64,f64,str,f64,f64,str,str,str,str,str,str,str,str,str,str,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,f64,f64,f64,…,f64,f64,f64,f64,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""count""","""13401""","""46403""",13448.0,13379.0,13379.0,"""46403""",46403.0,46403.0,"""46403""","""46403""","""13379""","""46403""","""46403""","""46403""","""46403""","""46403""","""46403""","""46403""","""46403""","""46403""",46403.0,46403.0,46403.0,13413.0,13413.0,13379.0,13379.0,46403.0,46403.0,46403.0,46403.0,46403.0,"""46403""",45980.0,45980.0,13414.0,…,46403.0,46403.0,46403.0,46403.0,"""9650""","""9650""",6831.0,46396.0,46396.0,0.0,0.0,4.0,46396.0,4.0,46403.0,46403.0,46396.0,46396.0,46403.0,46403.0,46403.0,46403.0,46403.0,46403.0,14197.0,14627.0,14681.0,14938.0,13413.0,13413.0,13413.0,1026.0,0.0,0.0,0.0,0.0,0.0
"""null_count""","""33002""","""0""",32955.0,33024.0,33024.0,"""0""",0.0,0.0,"""0""","""0""","""33024""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""",0.0,0.0,0.0,32990.0,32990.0,33024.0,33024.0,0.0,0.0,0.0,0.0,0.0,"""0""",423.0,423.0,32989.0,…,0.0,0.0,0.0,0.0,"""36753""","""36753""",39572.0,7.0,7.0,46403.0,46403.0,46399.0,7.0,46399.0,0.0,0.0,7.0,7.0,0.0,0.0,0.0,0.0,0.0,0.0,32206.0,31776.0,31722.0,31465.0,32990.0,32990.0,32990.0,45377.0,46403.0,46403.0,46403.0,46403.0,46403.0
"""mean""",null,"""2020-09-25 14:19:09.908411""",88.891649,-0.760262,5.948197,null,597081.783441,591175.467815,null,null,null,null,null,null,null,null,null,null,null,null,0.395384,0.393078,2020.42549,-0.172264,0.775522,-0.060429,2.431616,62019.123074,129384.394931,215981.852316,0.875568,4.87309,null,128.01148,134.40598,2.161263,…,2.613085,2.455596,2.629744,2.438937,null,null,184.331723,0.001104,0.049605,null,null,0.072,-0.049605,88.0,-0.160033,0.025903,0.507567,0.528384,28.404607,27.601534,28.923108,28.125229,1.345754,1.10258,5.549553,2.867095,5.773653,2.892958,2.176945,0.538734,-0.113857,38.52076,null,null,null,null,null
"""std""",null,null,5.625679,1.94483,0.516026,null,81035.954482,82385.943056,null,null,null,null,null,null,null,null,null,null,null,null,0.772084,0.689379,3.022541,0.895589,0.674902,0.577575,0.627593,185545.682854,248960.316563,289689.038675,0.811664,2.594383,null,36.688024,45.266168,6.057867,…,2.896515,2.742098,2.82591,2.813774,null,null,59.648929,0.058747,0.466649,null,null,0.019883,0.466649,0.0,3.371403,3.375099,0.295112,0.293841,3.712929,3.735127,3.71963,3.751928,0.607354,1.121145,6.419278,6.681005,6.543695,6.672886,1.010706,0.735867,0.904871,14.790172,null,null,null,null,null
"""min""",null,"""2015-04-05""",38.1,-4.66,-0.13,"""Abad, Fernando""",112526.0,112526.0,null,null,null,"""A.J. Burnett out on a sacrific…",null,null,null,null,null,null,null,null,0.0,0.0,2015.0,-2.78,-2.19,-2.295108,-1.009845,0.0,0.0,0.0,0.0,1.0,null,4.63,3.27,-17.386511,…,0.0,0.0,0.0,0.0,null,null,0.0,-0.739,-0.57,null

In [53]:
df_sc.with_columns(
    df_sc.with_columns(
        pl.all().is_null()
    ).collect().sum_horizontal().alias('null_count')
).group_by(
    'game_pk'
).agg(
    pl.col('null_count').mean().alias('mean_null_count'),
    pl.col('null_count').sum().alias('total_null_count')
).describe()

statistic,game_pk,mean_null_count,total_null_count
str,f64,f64,f64
"""count""",26713.0,26713.0,26713.0
"""null_count""",0.0,0.0,0.0
"""mean""",604666.378355,18.69876,5364.970277
"""std""",118948.604531,5.602438,1199.699472
"""min""",413649.0,12.370192,1975.0
"""25%""",491848.0,17.279693,4648.0
"""50%""",632219.0,18.5125,5277.0
"""75%""",717595.0,19.212928,5909.0
"""max""",813074.0,53.031546,22144.0


In [ ]:
df_sc.with_columns(
    df_sc.collect().is_null().sum_horizontal().alias('null_count')
).group_by(
    'game_pk'
).agg(

)

In [31]:
df_sc.filter(
    ~pl.col('hit_location').is_null() &
    (pl.col('type') != 'X') &
    (pl.col('hit_location') == "2") &
    (pl.col('events') != 'strikeout') &
    (pl.col('bb_type') != 'not_in_play')
).select(
    'des',
    'events',
    'description',
    'type',
    'hit_location',
    'bb_type'
).collect()

des,events,description,type,hit_location,bb_type
str,str,str,str,str,str
"""Lorenzo Cain reaches on catche…","""catcher_interf""","""hit_into_play""","""S""","""2""","""ground_ball"""
"""Tommy La Stella reaches on cat…","""catcher_interf""","""hit_into_play""","""S""","""2""","""line_drive"""
"""Nick Senzel reaches on catcher…","""catcher_interf""","""hit_into_play""","""S""","""2""","""ground_ball"""
"""Josh Reddick reaches on catche…","""catcher_interf""","""hit_into_play""","""S""","""2""","""ground_ball"""
"""Cavan Biggio reaches on catche…","""catcher_interf""","""hit_into_play""","""S""","""2""","""ground_ball"""
…,…,…,…,…,…
"""Travis Jankowski reaches on ca…","""catcher_interf""","""hit_into_play""","""S""","""2""","""ground_ball"""
"""Sal Frelick reaches on catcher…","""catcher_interf""","""hit_into_play""","""S""","""2""","""ground_ball"""
"""Kerry Carpenter reaches on cat…","""catcher_interf""","""hit_into_play""","""S""","""2""","""ground_ball"""


In [21]:
df_sc.filter(
    pl.col('hit_location').is_null()
).group_by('type').len().collect()

type,len
cat,u32


In [ ]:
df_sc.select(
    cs.matches('hc_[xy]'),
    'hit_distance_sc'
).drop_nulls().collect().sample(1000)


In [ ]:
df_sc.with_columns(
    (np.atan(
        (pl.col('hc_x')-125.42)/(198.27 - pl.col('hc_y'))
    )* 180 /np.pi*.5).alias('spray_angle')
).drop_nulls().select(
    cs.matches('hc_[xy]'),
    'spray_angle'
).describe()

In [ ]:
df_sc.filter(
    pl.col('hc_x').is_null().xor(pl.col('hit_distance_sc').is_null()) &
    (pl.col('des') != 'foul')
).select(
    'des',
    'type',
    'description',
    cs.matches('hc_[xy]'),
    'hit_distance_sc',
    cs.matches('launch'),
    'bat_speed'
).describe()

In [15]:
df_sc.filter(
    (pl.col('des') == 'foul')
).group_by(
    pl.col('hit_distance_sc').is_null(),
    pl.col('game_year')
).agg(
    pl.len()
).collect()

hit_distance_sc,game_year,len
bool,u16,u32
false,2024,118960
true,2024,16609
false,2023,118588
false,2016,65156
false,2015,25978
…,…,…
true,2016,60820
true,2021,16634
true,2018,49736


In [36]:
df = df_sc.filter(
    pl.col('type') == 'X'
).select(
    cs.matches('hc_[xy]'),
    cs.matches('distance'),
    pl.col('hit_location'),
    cs.matches('launch'),
    'bat_speed'
).drop_nulls().collect().sample(100000).to_dummies(cs.categorical())

In [37]:

#y = ['hc_x','hc_y']
y = ['hit_distance_sc']
X = [col for col in df.columns if col not in y]


In [38]:


from sklearn.model_selection import train_test_split as tts
X_train, X_test, y_train, y_test = tts(df[X], df[y], test_size=.2)

In [39]:
from sklearn.linear_model import LinearRegression as lr
from sklearn.preprocessing import PolynomialFeatures as poly
from sklearn.preprocessing import StandardScaler as ss
from sklearn.pipeline import Pipeline

ppl = Pipeline([('Scaler', ss()), ('PolynomialFeatures', poly(2)), ('LinearRegression', lr())])
ppl.fit(X_train, y_train)
ppl.score(X_test, y_test)

0.9512437386726164

In [41]:
from sklearn.neighbors import KNeighborsRegressor as knr
from sklearn.preprocessing import MinMaxScaler as mms
ppl = Pipeline([('Scaler', ss()), ('PolynomialFeatures', poly(1)), ('Regressor', knr())])

ppl.fit(X_train, y_train)
ppl.score(X_test, y_test)


0.983389253896015

In [42]:
df_sc.filter(
    pl.col('hit_distance_sc').is_null() & ~pl.col('hc_x').is_null()
).describe()

statistic,pitch_type,game_date,release_speed,release_pos_x,release_pos_z,player_name,batter,pitcher,events,description,zone,des,game_type,stand,p_throws,home_team,away_team,type,hit_location,bb_type,balls,strikes,game_year,pfx_x,pfx_z,plate_x,plate_z,on_3b,on_2b,on_1b,outs_when_up,inning,inning_topbot,hc_x,hc_y,vx0,…,post_away_score,post_home_score,post_bat_score,post_fld_score,if_fielding_alignment,of_fielding_alignment,spin_axis,delta_home_win_exp,delta_run_exp,bat_speed,swing_length,estimated_slg_using_speedangle,delta_pitcher_run_exp,hyper_speed,home_score_diff,bat_score_diff,home_win_exp,bat_win_exp,age_pit_legacy,age_bat_legacy,age_pit,age_bat,n_thruorder_pitcher,n_priorpa_thisgame_player_at_bat,pitcher_days_since_prev_game,batter_days_since_prev_game,pitcher_days_until_next_game,batter_days_until_next_game,api_break_z_with_gravity,api_break_x_arm,api_break_x_batter_in,arm_angle,attack_angle,attack_direction,swing_path_tilt,intercept_ball_minus_batter_pos_x_inches,intercept_ball_minus_batter_pos_y_inches
str,str,str,f64,f64,f64,str,f64,f64,str,str,str,str,str,str,str,str,str,str,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,f64,f64,f64,…,f64,f64,f64,f64,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""count""","""69430""","""102424""",69514.0,69439.0,69439.0,"""102424""",102424.0,102424.0,"""102424""","""102424""","""69439""","""102424""","""102424""","""102424""","""102424""","""102424""","""102424""","""102424""","""102424""","""102424""",102424.0,102424.0,102424.0,69466.0,69466.0,69439.0,69439.0,102424.0,102424.0,102424.0,102424.0,102424.0,"""102424""",102424.0,102424.0,69474.0,…,102424.0,102424.0,102424.0,102424.0,"""64911""","""64911""",50632.0,102417.0,102421.0,759.0,759.0,55061.0,102421.0,55777.0,102424.0,102424.0,102417.0,102417.0,102424.0,102424.0,102424.0,102424.0,102424.0,102424.0,67302.0,69383.0,68110.0,69607.0,69466.0,69466.0,69466.0,1836.0,759.0,759.0,759.0,757.0,757.0
"""null_count""","""32994""","""0""",32910.0,32985.0,32985.0,"""0""",0.0,0.0,"""0""","""0""","""32985""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""",0.0,0.0,0.0,32958.0,32958.0,32985.0,32985.0,0.0,0.0,0.0,0.0,0.0,"""0""",0.0,0.0,32950.0,…,0.0,0.0,0.0,0.0,"""37513""","""37513""",51792.0,7.0,3.0,101665.0,101665.0,47363.0,3.0,46647.0,0.0,0.0,7.0,7.0,0.0,0.0,0.0,0.0,0.0,0.0,35122.0,33041.0,34314.0,32817.0,32958.0,32958.0,32958.0,100588.0,101665.0,101665.0,101665.0,101667.0,101667.0
"""mean""",null,"""2018-12-26 01:47:53.420292""",88.816397,-0.772591,5.926961,null,561236.781789,560541.975719,null,null,null,null,null,null,null,null,null,null,null,null,0.784094,0.791416,2018.571829,-0.166502,0.700559,-0.033297,2.240291,57323.689311,114284.985443,192305.53177,0.929128,4.960185,null,127.98803,151.047078,2.232855,…,2.507332,2.363616,2.484925,2.386023,null,null,183.042878,0.000549,-0.063037,65.349407,6.804875,0.089832,0.063037,88.584295,-0.145718,-0.001845,0.50719,0.521721,28.366184,27.904563,28.881209,28.430046,1.461728,1.347057,5.649877,2.030483,5.771605,2.086213,2.26796,0.499495,-0.107426,38.119826,3.779564,3.161892,33.282533,37.756187,26.736677
"""std""",null,null,5.796885,1.931676,0.52571,null,88347.605223,85574.424748,null,null,null,null,null,null,null,null,null,null,null,null,0.974307,0.837475,2.958691,0.906305,0.703116,0.586258,0.668736,173050.493287,230166.783336,270978.85035,0.815734,2.627315,null,32.171051,39.824896,5.997003,…,2.772729,2.690584,2.714572,2.750237,null,null,66.31275,0.05085,0.396932,15.878164,1.523047,0.26414,0.396932,2.093174,3.290156,3.293381,0.294924,0.29421,3.747748,3.791519,3.762134,3.799705,0.708269,1.233822,6.926789,4.777094,7.375113,5.165518,1.039631,0.774347,0.915189,14.176335,16.12394,29.149371,8.918648,7.185624,10.62384
"""min""",null,"""2015-04-05""",38.1,-5.17,-0.13,"""Aardsma, David""",112526.0,112526.0,null,null,null,"""A.J. Burnett grounds out softl…",null,null,null,null,null,null,null,

In [89]:
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import cross_val_predict


class RegressorFeature(BaseEstimator, TransformerMixin):
    """
    Wraps a regressor and exposes its predictions as a single feature column.
    Uses cross_val_predict during fit to avoid leakage, then re-fits on full data
    for use at transform time.
    """

    def __init__(self, estimator, cv=5):
        self.estimator = estimator
        self.cv = cv

    def fit(self, X, y):
        self.estimator_ = clone(self.estimator)
        # Re-fit on full data so transform() works on new data
        self.estimator_.fit(X, y)
        return self

    def transform(self, X):
        preds = self.estimator_.predict(X)
        return preds.reshape(-1, 2)

    def fit_transform(self, X, y=None, **fit_params):
        self.estimator_ = clone(self.estimator)
        # Use OOF predictions during training to prevent leakage
        oof_preds = cross_val_predict(self.estimator_, X, y, cv=self.cv)
        self.estimator_.fit(X, y)
        return oof_preds.reshape(-1, 2)

In [93]:
from xgboost import XGBRegressor as xgb
from sklearn.model_selection import GridSearchCV as gcv
from sklearn.pipeline import FeatureUnion as union
ppl = Pipeline([('Scaler', mms()),
                ('PolynomialFeatures', poly(degree=3)),
                ('Scaler2', mms()),
                ('PCA', PCA(n_components=50)),
                ("features", union([
                   # Original features (scaled)
                    ("original", ss()),
                    # New meta-feature: Ridge predictions
                    ("ridge_pred", RegressorFeature(estimator=knr(n_neighbors=5), cv=5)),
                 ])),
                ('Regressor', xgb(n_estimators=200, max_leaves=4, device='cuda'))])

cv = gcv(ppl,
         param_grid={
             'PolynomialFeatures__degree': [1,2],
             'PCA__n_components': [5,10,15],
             'Regressor__n_estimators': [25,50]

         }, cv = 3, verbose=3)

cv.fit(X_train, y_train)
cv.score(X_test, y_test)


Fitting 3 folds for each of 12 candidates, totalling 36 fits
[CV 1/3] END PCA__n_components=5, PolynomialFeatures__degree=1, Regressor__n_estimators=25;, score=0.812 total time=   0.9s
[CV 2/3] END PCA__n_components=5, PolynomialFeatures__degree=1, Regressor__n_estimators=25;, score=0.816 total time=   0.8s
[CV 3/3] END PCA__n_components=5, PolynomialFeatures__degree=1, Regressor__n_estimators=25;, score=0.817 total time=   0.7s
[CV 1/3] END PCA__n_components=5, PolynomialFeatures__degree=1, Regressor__n_estimators=50;, score=0.815 total time=   0.9s
[CV 2/3] END PCA__n_components=5, PolynomialFeatures__degree=1, Regressor__n_estimators=50;, score=0.819 total time=   0.8s
[CV 3/3] END PCA__n_components=5, PolynomialFeatures__degree=1, Regressor__n_estimators=50;, score=0.819 total time=   0.7s
[CV 1/3] END PCA__n_components=5, PolynomialFeatures__degree=2, Regressor__n_estimators=25;, score=0.812 total time=   0.7s
[CV 2/3] END PCA__n_components=5, PolynomialFeatures__degree=2, Regress

0.8179455995559692

In [85]:
cv.best_params_

{'PCA__n_components': 50,
 'PolynomialFeatures__degree': 3,
 'Regressor__n_estimators': 200}

In [1]:
from sklearn.neural_network import MLPRegressor as mlp
from sklearn.decomposition import PCA
ppl = Pipeline([('Scaler', mms()),
                ('PolynomialFeatures', poly(3)),
                ('Scaler2', mms()),('PCA', PCA(400)),
                ('Regressor', mlp(hidden_layer_sizes=(18, 18),
                                  learning_rate='adaptive',
                                  max_iter=1000,
                                  verbose = True,
                                  early_stopping=True,
                                  n_iter_no_change=50,))])

ppl.fit(X_train, y_train)
ppl.score(X_test, y_test)

ModuleNotFoundError: No module named 'sklearn'

In [95]:
pd.concat([pd.DataFrame(ppl.predict(X_test)), pd.DataFrame(y_test)], axis=1)

,0,1,0,1
0,116.308687,122.801529,51.85,127.94
1,112.537247,149.804064,116.32,149.52
2,114.485353,155.260477,115.05,159.04
3,64.407365,90.625959,70.55,94.31
4,112.516453,149.452030,114.61,145.00
...,...,...,...,...
19995,110.410314,150.067502,106.59,150.63
19996,102.816344,165.397289,109.08,174.53
19997,111.955658,150.672915,112.82,163.35
19998,112.260008,154.040449,138.78,150.90


In [54]:
from sklearn.linear_model import MultiTaskElasticNetCV as ecv
regr = ecv(cv=10, verbose=True)
regr.fit(X_train, y_train)
regr.score(X_test, y_test)

........................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................

0.41420841843736295

In [56]:
regr.coef_

array([[-0.0053954 ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        , -0.74594874,  0.        ,  0.789843  ,
         0.        ,  0.        ,  0.        ,  0.        , -0.03319168,
         0.05472859,  0.        ],
       [-0.2798944 ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        , -0.00562719, -0.        ,  0.00800857,
         0.        ,  0.        ,  0.        ,  0.        , -0.52633608,
         0.27489877,  0.        ]])

In [17]:
df_sc.group_by(
    'game_pk'
).agg(
    pl.col('pitch_type').is_null().sum().alias('null_pitch_count'),
    pl.col('game_pk').len().alias('total_pitches'),
    pl.col('game_year').max()
).group_by(
    (pl.col('null_pitch_count')/pl.col('total_pitches') == 1).alias('epic_statcast_failure')
).agg(
    pl.col('null_pitch_count').sum()
).collect()

epic_statcast_failure,null_pitch_count
bool,u32
false,30381
true,106050


In [18]:
import altair as alt
alt.data_transformers.enable("vegafusion")
df = df_sc.group_by(
    'game_pk'
).agg(
    pl.col('release_speed').is_null().sum().alias('null_count'),
    pl.col('game_pk').len().alias('total_pitches'),
    pl.col('game_year').max()
).with_columns(
    (pl.col('null_count')/pl.col('total_pitches')).alias('null_fraction'),
    (pl.col('null_count')/pl.col('total_pitches') == 1).alias('epic_failure')
).filter(
    ~pl.col('epic_failure')
).collect()



In [19]:

chart = alt.Chart(
    df
).mark_bar(

).encode(
    alt.X(
        "null_fraction:Q",
        bin=True,
    ),
    alt.Y('count()').scale(type = 'log')
)
chart

alt.Chart(...)